# Frozen ConvNet branch — produce / inspect the DeepConvLSTM predictions used in the fusion

The locked model fuses the GNN branch with a **frozen** ConvNet (DeepConvLSTM, frequency view,
`run2_nohann` recipe): `P = 0.6·P_gnn + 0.4·P_conv`. The ConvNet is trained **once per seed, per LODO
fold**, and its target-test probabilities are saved as `convnet_<target>.npz` and reused unchanged by
every GNN configuration (P21) and every SO operating point (P24).

| Folder | What it holds |
|---|---|
| `results/sweep/base_s{42,1,2}/convnet_*.npz` | **The original frozen predictions used in the thesis** (from the P18b source-only run). Never overwritten by this notebook. |
| `results/sweep/base_s*/gnn_*.npz` | The fixed-topology GNN (Arm A) of the same run — kept only because the files were produced together. |
| `results/sweep_rerun/base_s<seed>/` | Where **this** notebook writes any re-trained ConvNet predictions. |

Seeds 3 and 5 of the thesis's 5-seed protocol reused the seed-42 ConvNet predictions (stand-in); §5 below
can optionally train genuine seed-3/5 ConvNets.

**Nothing here needs to be re-run to reproduce the thesis numbers** — §3 just inspects the saved files.

## 1. Mount Drive & paths

In [ ]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("convnet_b")   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH
FROZEN = PP.CONVNET_FROZEN_SWEEP   # original thesis predictions (read-only)
RERUN  = PP.CONVNET_RERUN_SWEEP    # output of any re-run in this notebook
TARGETS = PP.DATASETS
assert os.path.isdir(FROZEN), FROZEN

## 2. Setup

In [ ]:
import setup_colab
setup_colab.install_deps(); assert setup_colab.verify(), "fix DATA_PATH in project_paths.py"

## 3. Inspect the frozen predictions (no GPU, instant)

In [ ]:
import numpy as np
def acc_table(root, seeds):
    rows = {}
    for s in seeds:
        d = os.path.join(root, f"base_s{s}")
        accs = {}
        for t in TARGETS:
            f = os.path.join(d, f"convnet_{t}.npz")
            if os.path.isfile(f):
                z = np.load(f); accs[t] = (z["probs"].argmax(1) == z["labels"]).mean() * 100
        rows[s] = accs
    print(f"{'seed':>5s} " + " ".join(f"{t[:10]:>10s}" for t in TARGETS) + f" {'MEAN':>7s}")
    for s, a in rows.items():
        if not a: print(f"{s:>5} (no files)"); continue
        print(f"{s:>5} " + " ".join(f"{a.get(t, float('nan')):10.2f}" for t in TARGETS) + f" {np.mean(list(a.values())):7.2f}")
    return rows
frozen = acc_table(FROZEN, [42, 1, 2])

## 4. Smoke test (~2 min)
One fold, 2 epochs, written to `results/_smoke` (then deleted).

In [ ]:
!python -m da_baseline.convnet_predict --data_path "$DATA_PATH" --targets WISDM --epochs 2 --log_every 1 --seed 42 --out_dir results/_smoke
import numpy as np, shutil
z = np.load("results/_smoke/convnet_WISDM.npz"); print("smoke OK:", z["probs"].shape, z["labels"].shape)
shutil.rmtree("results/_smoke")

## 5. (Optional) Re-train the ConvNet — ~10–15 min per seed on a T4
Writes to `results/sweep_rerun/base_s<seed>/`; folds whose file already exists are skipped, so a disconnect
only re-runs what's left. Retrained predictions will differ slightly from the frozen ones (GPU
non-determinism), so the thesis numbers keep using the frozen files unless you deliberately copy a re-run
into `P21_PearsonGraph/results/sweep/base_s<seed>/`.

Set `SEEDS = [3, 5]` to train genuine seed-3/5 ConvNets (the thesis reused seed 42 for those seeds).

In [ ]:
SEEDS = [42, 1, 2]        # e.g. [3, 5] for genuine seed-3/5 ConvNets
for s in SEEDS:
    out = os.path.join(RERUN, f"base_s{s}")
    os.makedirs(out, exist_ok=True)
    for t in TARGETS:
        if os.path.isfile(os.path.join(out, f"convnet_{t}.npz")):
            print(f"[skip] seed {s} {t} already done"); continue
        !python -m da_baseline.convnet_predict --data_path "$DATA_PATH" --targets {t} --epochs 100 --log_every 10 --seed {s} --out_dir "{out}"

## 6. Compare re-run vs frozen

In [ ]:
rerun = acc_table(RERUN, SEEDS)
for s in SEEDS:
    if s in frozen and frozen[s] and rerun.get(s):
        common = [t for t in TARGETS if t in frozen[s] and t in rerun[s]]
        d = np.mean([rerun[s][t] for t in common]) - np.mean([frozen[s][t] for t in common])
        print(f"seed {s}: re-run − frozen = {d:+.2f} pp over {len(common)} folds")